In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"Using compute device: {device}")


In [ ]:
# Setup paths and transform
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET_DIR = PROJECT_ROOT / "dataset" / "cats_and_dogs"
MODEL_PATH = PROJECT_ROOT / "models" / "cat_dog_model.pth"

# Exact same transform used during training
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

class_names = ['cats', 'dogs']
print("Model Path:", MODEL_PATH)
print("Classes:", class_names)


In [ ]:
# Simple CNN Architecture
class SimpleCatDogCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        # PART 1: Find Clues (2 simple scanning rounds)
        self.features = nn.Sequential(
            # Round 1: 16 filters | shrinks 128x128 -> 64x64
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            # Round 2: 32 filters | shrinks 64x64 -> 32x32
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        # PART 2: Make the Final Guess
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 32, 2)  # Vote: Cat (0) or Dog (1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Load trained model weights from disk
model = SimpleCatDogCNN().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print("Model weights loaded successfully from disk!")


In [ ]:
def review_prediction(image_path):
    img = Image.open(image_path)
    tensor = transform(img.convert("RGB")).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(tensor)
        probs = torch.softmax(output, dim=1)[0]
        pred_idx = probs.argmax().item()
        
    pred_label = class_names[pred_idx]
    confidence = probs[pred_idx].item() * 100
    
    plt.figure(figsize=(3.5, 3.5))
    plt.imshow(img)
    plt.title(f"Guess: {pred_label.upper()} ({confidence:.1f}%)", fontsize=12)
    plt.xlabel(f"Cat: {probs[0]*100:.1f}% | Dog: {probs[1]*100:.1f}%", fontsize=10)
    plt.xticks([])
    plt.yticks([])
    plt.show()

# Review sample photos from the validation folder
cats = list((DATASET_DIR / "validation" / "cats").glob("*.jpg"))[:2]
dogs = list((DATASET_DIR / "validation" / "dogs").glob("*.jpg"))[:2]

for cat_path in cats:
    review_prediction(cat_path)

for dog_path in dogs:
    review_prediction(dog_path)
